# babaorum — R7(a) S3 : treillis A1 (T4, fp16)

docs/READER_R7A_PREREG.txt (Q3) ; contrôles d'entrée AVANT toute lecture ; « GPU T4 x2 », Internet ON ; jeu privé babaorum-r7a-s3-inputs attaché.

In [ ]:
import os, sys, json, gzip, glob, hashlib, ast, time, math, unicodedata
T_START = time.time()
INPUTS_SHA = "06ed420cbb074aeb93e179cd0ea150c26189a3139dd93c58ed6bb630ac3c48b7"
EXCL_SHA = "efeebff4a605735a6e8ec049fafc52dfdb4be97ee49e3d340fd452babfe2692e"
FREEZE_SHA = "53e17a2bcb78133315ac4d309d03e39ae71e37d7c4fc8a53ab410bad182e42e7"
REPO, REV = "dots-studio/dots.mocr", "e539fbb52280393adc081b289ec597430a0f9031"
L, HEIGHT, MAXW, UP, BS = 32, 56, 2800, 2, 32
OUT, TMP = "/kaggle/working/r7a_s3", "/kaggle/tmp"
PLAFOND_S = 5 * 3600
assert not OUT.startswith("/kaggle/input"), "sortie dans les entrées"
os.makedirs(OUT, exist_ok=True); os.makedirs(TMP, exist_ok=True)
assert not os.listdir(OUT), "dossier de sortie non vide : relancer dans une session neuve"
class Arret(Exception):
    pass
def need(c, m):
    if not c:
        raise Arret(m)
def h(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()
mf = glob.glob("/kaggle/input/**/MANIFEST.json", recursive=True)
need(len(mf) == 1, f"un seul MANIFEST.json attendu dans /kaggle/input : {mf}")
D = os.path.dirname(mf[0])
need(h(mf[0]) == INPUTS_SHA, "MANIFEST du jeu différent de INPUTS_SHA")
MAN = json.load(open(mf[0]))
need(set(MAN) == set(os.listdir(D)) - {"MANIFEST.json", "dataset-metadata.json"}, "fichiers du jeu en plus ou en moins")
for n, s in MAN.items():
    need(h(f"{D}/{n}") == s, f"fichier du jeu modifié : {n}")
# chaîne exclusion -> constructeur -> sources, AVANT toute lecture
need(h(f"{D}/excl_MANIFEST.json") == EXCL_SHA, "MANIFEST d'exclusion différent du SHA épinglé")
EXM = json.load(open(f"{D}/excl_MANIFEST.json"))
need(h(f"{D}/items_retenus.jsonl") == EXM["sorties"]["items_retenus.jsonl"], "items_retenus.jsonl n'est pas celui de l'exclusion")
need(h(f"{D}/build_MANIFEST.json") == EXM["entrees"]["build_MANIFEST.json"], "MANIFEST du constructeur non cité par l'exclusion")
BM = json.load(open(f"{D}/build_MANIFEST.json"))
need(h(f"{D}/build_sources.json") == BM["sorties"]["sources.json"], "sources.json n'est pas celui du constructeur")
SOURCES = json.load(open(f"{D}/build_sources.json"))
PAGE_FILE = {}
for pk, s in SOURCES.items():
    if s.get("refusee"):
        continue
    f = [n for n in MAN if n.startswith(f"page_{s['image_sha256']}.")]
    need(len(f) == 1 and MAN[f[0]] == s["image_sha256"], f"image absente ou différente : {pk}")
    PAGE_FILE[s["image"]] = f"{D}/{f[0]}"
# gel : document épinglé, 9 empreintes, valeurs LUES et vérifiées
GEL = f"{D}/gel__docs__READER_DOTS_FREEZE_R7A.txt"
need(h(GEL) == FREEZE_SHA, "document du gel différent de FREEZE_SHA")
import re
ent = {m.group(2): m.group(1) for m in (re.match(r"\s+([0-9a-f]{64})\s+(\S+)", l) for l in open(GEL, encoding="utf-8")) if m}
need(len(ent) == 9, "9 empreintes de gel attendues")
for p, s in ent.items():
    need(h(f"{D}/gel__{p.replace('/', '__')}") == s, f"gel rompu : {p}")
TAU = json.load(open(f"{D}/gel__reports__reader_dots_abst_v2.json"))["tau"]
LAMBDA = json.load(open(f"{D}/gel__reports__reader_dots_p2_tune.json"))["choix"]["lambda_star"]
need(TAU == 20.827896720535783 and LAMBDA == 0.3, f"valeurs gelées inattendues : tau {TAU}, lambda {LAMBDA}")
src = open(f"{D}/gel__scripts__reader_dots_p3_report.py", encoding="utf-8").read()
keep = [n for n in ast.parse(src).body if (isinstance(n, ast.Assign) and any(getattr(t, "id", "") == "_NORM" for t in n.targets))
        or (isinstance(n, ast.FunctionDef) and n.name == "norm")]
need(len(keep) == 2, "NORM introuvable dans reader_dots_p3_report.py")
NS = {"unicodedata": unicodedata}
exec(compile(ast.Module(body=keep, type_ignores=[]), "p3_report:NORM", "exec"), NS)
NORM = NS["norm"]
need(NORM("l’a  «b» —c") == "l'a \"b\" -c", "NORM ne se comporte pas comme le gel")
sys.path.insert(0, D)
ITEMS = [json.loads(l) for l in open(f"{D}/items_retenus.jsonl", encoding="utf-8")]
need(len({i["stem"] for i in ITEMS}) == len(ITEMS), "tiges en double")
print(len(ITEMS), "items ;", len(PAGE_FILE), "pages ; tau", TAU, "; lambda*", LAMBDA, "; contrôles d'entrée OK")

In [ ]:
%%bash
set -e
pip install -q safetensors huggingface_hub 2>&1 | tail -1
python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

In [ ]:
import torch, shutil
from safetensors import safe_open
from huggingface_hub import hf_hub_download
need(torch.cuda.is_available() and torch.cuda.get_device_capability()[0] == 7, "T4 attendu (fp16)")
DT = torch.float16
from vit import NaViT, SPECS, MEAN, STD, prep, rename_map
from ctc import Head, read_lines, white_value
from crops import crop_line
need((REPO, REV) == (SPECS["dots_mocr"]["repo"], SPECS["dots_mocr"]["rev"]), "révision de dots.mocr")
idx = json.load(open(hf_hub_download(REPO, "model.safetensors.index.json", revision=REV, cache_dir=f"{TMP}/hf")))
s = SPECS["dots_mocr"]
ren = rename_map(s["prefix"], L)
sd = {}
for shard in sorted({idx["weight_map"][k] for k in ren}):
    p = hf_hub_download(REPO, shard, revision=REV, cache_dir=f"{TMP}/hf")
    with safe_open(p, framework="pt") as fh:
        for k in fh.keys():
            if k in ren:
                sd[ren[k]] = fh.get_tensor(k)
    os.remove(os.path.realpath(p))
shutil.rmtree(f"{TMP}/hf", ignore_errors=True)
enc = NaViT(s["dim"], s["heads"], s["ffn"], L, s["rope"])
missing, unexpected = enc.load_state_dict(sd, strict=False)
need(not unexpected and [k for k in missing if k != "inv_freq"] == [], "poids de l'encodeur")
enc = enc.to("cuda", DT).eval(); del sd
TIMING = json.load(open(f"{D}/timing.json"))
need(TIMING.get("L") == L, "timing.json n'est pas celui d'A1 (L = 32)")
CHARS = TIMING["chars"]
head = Head(len(CHARS) + 1, up=UP)
head.load_state_dict(torch.load(f"{D}/head_a1.pt", map_location="cpu"))
head = head.to("cuda").eval()
WHITE = white_value(MEAN, STD)
print("encodeur", L, "blocs", DT, "; tête", len(CHARS), "caractères")

In [ ]:
import numpy as np
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
CROP, ecarts = {}, []
by_img = {}
for it in ITEMS:
    by_img.setdefault(it["image"], []).append(it)
for img, group in sorted(by_img.items()):
    need(img in PAGE_FILE, f"page non gardée ou inconnue : {img}")
    with Image.open(PAGE_FILE[img]) as im:
        page = im.convert("L")
    for it in group:
        c = crop_line(page, tuple(it["box"]), it["polygon"], pad=4).image
        sig = float(np.asarray(c, dtype=np.float64).std())
        if (hashlib.md5(c.tobytes()).hexdigest() != it["crop_md5"] or abs(sig - it["sigma"]) > 1e-6
                or (sig < TAU) != it["abstention"]):
            ecarts.append(it["stem"])
        CROP[it["stem"]] = c
need(not ecarts, f"recadrages différents du constructeur : {len(ecarts)} ({ecarts[:3]})")
print(len(CROP), "recadrages identiques au constructeur (MD5, sigma, abstention)")

In [ ]:
amp = lambda: torch.autocast("cuda", dtype=DT)
order = sorted(ITEMS, key=lambda it: CROP[it["stem"]].width * HEIGHT / max(1, CROP[it["stem"]].height))
rows, non_finis = [], []
with torch.no_grad():
    for i in range(0, len(order), BS):
        need(time.time() - T_START < PLAFOND_S, "plafond de temps atteint")
        b = order[i:i + BS]
        for it, (hyp, lp) in zip(b, read_lines(enc, head, [prep(CROP[it["stem"]], HEIGHT, MAXW) for it in b], L, CHARS, WHITE, "cuda", amp)):
            lp16 = lp.float().cpu().half()  # mineur 1 Sonnet : finitude contrôlée sur ce qui est écrit (float16)
            if not torch.isfinite(lp16).all():
                non_finis.append(it["stem"])
            rows.append({"stem": it["stem"], "part": it["part"], "hyp": hyp, "logp": lp16.numpy().tolist()})
need(not non_finis, f"logp non fini en fp16 : {len(non_finis)} lignes ({non_finis[:3]}) ; arrêt, retour à pA")
need(len(rows) == len(ITEMS), "lignes manquantes")
with gzip.open(f"{OUT}/a1.jsonl.gz", "wt", encoding="utf-8") as g:
    for r in sorted(rows, key=lambda r: r["stem"]):
        g.write(json.dumps(r, ensure_ascii=False) + "\n")
journal = {"items": len(rows), "par_part": {p: sum(r["part"] == p for r in rows) for p in sorted({r["part"] for r in rows})},
           "abstentions_prevues": sum(it["abstention"] for it in ITEMS), "dtype": str(DT), "gpu": torch.cuda.get_device_name(0),
           "secondes": round(time.time() - T_START), "tau": TAU, "lambda_star": LAMBDA, "norm_verifiee": True}
json.dump(journal, open(f"{OUT}/journal.json", "w"), indent=1, ensure_ascii=False)
man = {"entrees": {"INPUTS_SHA": INPUTS_SHA, "EXCL_SHA": EXCL_SHA, "FREEZE_SHA": FREEZE_SHA, "REV": REV},
       "sorties": {n: h(f"{OUT}/{n}") for n in sorted(os.listdir(OUT))}}
json.dump(man, open(f"{OUT}/MANIFEST.json", "w"), indent=1)
print(json.dumps(journal, ensure_ascii=False))